# Kenya Counties Rainfall Analysis & Planting Advisory

**Student Name:** Peter Kuria  
**Student Email:** peakaykush@gmail.com  
**Cohort:** 1 milliondevs 2026 Data analytics  
**Organization:** Farmer-Support NGO  
**Assignment:** Assignment 2 — NumPy Rainfall Data Cleaning & Advisory  

---

In [1]:
# Dataset provided for the five counties
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

## Task 1: Clean and Impute Machakos Rainfall Data

**Approach:**  
The rainfall readings for Machakos County arrived as raw strings containing whitespaces, missing strings (`""`), and non-numeric placeholders (`"N/A"`).  
We construct a cleaning function `clean_machakos(raw_data, month_names)` that:
1. Uses a specific `try...except ValueError` block to parse valid integers.
2. Calculates the rounded arithmetic average of all valid readings.
3. Replaces invalid/missing entries with this rounded average.
4. Identifies and prints which calendar months required imputation.

In [2]:
def clean_machakos(raw_data, month_names):
    """Clean Machakos raw rainfall data using try/except ValueError.
    
    Validates string values, converts valid values to integers, and replaces
    invalid entries with the rounded average of valid readings.
    
    Args:
        raw_data (list): List of raw monthly rainfall strings.
        month_names (list): List of corresponding month names.
        
    Returns:
        list: Cleaned list of 12 monthly rainfall integers.
    """
    valid_values = []
    imputed_indices = []
    
    # Pass 1: Extract valid integers and log indices needing imputation
    for idx, item in enumerate(raw_data):
        try:
            valid_values.append(int(item.strip()))
        except ValueError:
            imputed_indices.append(idx)
            
    # Calculate the rounded average of valid readings
    avg_val = round(sum(valid_values) / len(valid_values))
    
    # Pass 2: Reconstruct the full 12-month sequence
    cleaned_row = []
    for idx, item in enumerate(raw_data):
        try:
            cleaned_row.append(int(item.strip()))
        except ValueError:
            cleaned_row.append(avg_val)
            
    # Print the imputed months as required
    imputed_month_names = [month_names[i] for i in imputed_indices]
    print(f"Valid readings count    : {len(valid_values)}")
    print(f"Rounded average rainfall: {avg_val} mm")
    print(f"Imputed months          : {', '.join(imputed_month_names)} (Indices: {imputed_indices})")
    
    return cleaned_row

# Execute cleaning function for Machakos
machakos_clean = clean_machakos(machakos_raw, months)
print(f"Cleaned Machakos series : {machakos_clean}")

Valid readings count    : 10
Rounded average rainfall: 68 mm
Imputed months          : Apr, Sep (Indices: [3, 8])
Cleaned Machakos series : [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Task 2: Combine County Data into a 5 × 12 NumPy Array

**Approach:**  
We combine the four clean county rows (`clean_rows`) with the newly cleaned `machakos_clean` row into a single 2D Python list. We convert this list into a NumPy array `R` and explicitly print its shape `(5, 12)` and data type to confirm proper alignment.

In [3]:
import numpy as np

# Combine all rows into a single 5 x 12 NumPy array
all_data = clean_rows + [machakos_clean]
rainfall_array = np.array(all_data)

# Print shape and dtype
print(f"Array Shape : {rainfall_array.shape}")
print(f"Array Dtype : {rainfall_array.dtype}")
print("\nRainfall Array (5 counties x 12 months, in mm):")
print(rainfall_array)

Array Shape : (5, 12)
Array Dtype : int64

Rainfall Array (5 counties x 12 months, in mm):
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## Task 3: Annual County Totals and Monthly Averages (NumPy Axis Operations)

**Approach:**  
Without using manual Python loops over the array, we compute:
- **Annual total rainfall per county:** Using `np.sum(rainfall_array, axis=1)` to sum horizontally across all 12 months for each county.
- **Monthly average rainfall across all counties:** Using `np.mean(rainfall_array, axis=0)` to compute the average down columns across the 5 counties for each calendar month.

In [4]:
# Annual total rainfall per county (sum along axis 1)
annual_totals = np.sum(rainfall_array, axis=1)

# Average rainfall per month across all 5 counties (mean along axis 0)
monthly_averages = np.mean(rainfall_array, axis=0)

print("Annual Total Rainfall per County:")
for county, total in zip(counties, annual_totals):
    print(f"  - {county:<10}: {total:>5} mm")

print("\nAverage Monthly Rainfall Across All Counties:")
for month, avg in zip(months, monthly_averages):
    print(f"  - {month}: {avg:6.2f} mm")

Annual Total Rainfall per County:
  - Nairobi   :  1045 mm
  - Nakuru    :   916 mm
  - Kisumu    :  1371 mm
  - Mombasa   :  1075 mm
  - Machakos  :   811 mm

Average Monthly Rainfall Across All Counties:
  - Jan:  47.00 mm
  - Feb:  47.60 mm
  - Mar:  91.00 mm
  - Apr: 164.40 mm
  - May: 171.20 mm
  - Jun:  67.60 mm
  - Jul:  56.60 mm
  - Aug:  58.20 mm
  - Sep:  64.20 mm
  - Oct:  67.80 mm
  - Nov: 127.40 mm
  - Dec:  80.60 mm


## Task 4: Wettest Month per County (`np.argmax` with axis=1)

**Approach:**  
We identify the peak rainfall month for each county using `np.argmax(rainfall_array, axis=1)`. This vector operation returns the column index of the maximum rainfall value for each county row. We then use `zip()` to pair county names with their wettest month and maximum rainfall amount.

In [5]:
# Find wettest month index per county using np.argmax with axis=1
wettest_indices = np.argmax(rainfall_array, axis=1)
wettest_months = [months[idx] for idx in wettest_indices]
max_rainfalls = np.max(rainfall_array, axis=1)

print("Wettest Month per County:")
for county, month, amount in zip(counties, wettest_months, max_rainfalls):
    print(f"  - {county:<10}: {month} ({amount} mm)")

Wettest Month per County:
  - Nairobi   : Apr (219 mm)
  - Nakuru    : Apr (145 mm)
  - Kisumu    : Apr (240 mm)
  - Mombasa   : May (240 mm)
  - Machakos  : Nov (182 mm)


## Task 5: Counting Dry Months (< 50 mm) using Boolean Masking

**Approach:**  
We assess drought exposure by constructing a boolean mask: `dry_mask = rainfall_array < 50`. We then count dry months per county by summing True values along `axis=1` (`np.sum(dry_mask, axis=1)`), avoiding any Python loops over elements.

In [6]:
# Boolean mask for dry months (< 50 mm)
dry_mask = rainfall_array < 50
dry_month_counts = np.sum(dry_mask, axis=1)

print("Dry Months Count (< 50 mm) per County:")
for county, count in zip(counties, dry_month_counts):
    print(f"  - {county:<10}: {count} dry months")

Dry Months Count (< 50 mm) per County:
  - Nairobi   : 4 dry months
  - Nakuru    : 3 dry months
  - Kisumu    : 0 dry months
  - Mombasa   : 2 dry months
  - Machakos  : 6 dry months


## Task 6: Optimal Planting Windows (Shifted-Array Technique)

**Approach:**  
To identify the two consecutive calendar months with the highest combined rainfall for each county without loops, we employ the vectorized shifted-array slice technique:
$$\text{two\_month\_sums} = R[:, :-1] + R[:, 1:]$$
- `R[:, :-1]` slices columns 0 through 10 (Jan to Nov).
- `R[:, 1:]` slices columns 1 through 11 (Feb to Dec).
The sum creates a `(5, 11)` array where column $k$ contains the sum of months $k$ and $k+1$. Applying `np.argmax(..., axis=1)` gives the starting index of the optimal planting window.

In [7]:
# Shifted-array technique to calculate 2 consecutive months combined rainfall
two_month_sums = rainfall_array[:, :-1] + rainfall_array[:, 1:]
best_window_indices = np.argmax(two_month_sums, axis=1)
max_window_totals = np.max(two_month_sums, axis=1)

planting_windows = [f"{months[i]}-{months[i+1]}" for i in best_window_indices]

print("Recommended 2-Month Planting Window per County:")
for county, window, total in zip(counties, planting_windows, max_window_totals):
    print(f"  - {county:<10}: {window} (Combined Rainfall: {total} mm)")

Recommended 2-Month Planting Window per County:
  - Nairobi   : Apr-May (Combined Rainfall: 395 mm)
  - Nakuru    : Apr-May (Combined Rainfall: 265 mm)
  - Kisumu    : Apr-May (Combined Rainfall: 420 mm)
  - Mombasa   : Apr-May (Combined Rainfall: 390 mm)
  - Machakos  : Nov-Dec (Combined Rainfall: 278 mm)


## Task 7: Generate `rainfall_summary.csv`

**Approach:**  
We assemble the analyzed metrics for all five counties and export them into `rainfall_summary.csv` using Python's standard `csv` module. The CSV file contains exactly 5 rows with columns: `county`, `annual_total_mm`, `wettest_month`, `dry_months`, and `planting_window`.

In [8]:
import csv

csv_filename = "rainfall_summary.csv"
fieldnames = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

# Write CSV file with exactly 5 rows using the csv module
with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for county, total, wettest, dry, window in zip(counties, annual_totals, wettest_months, dry_month_counts, planting_windows):
        writer.writerow({
            "county": county,
            "annual_total_mm": int(total),
            "wettest_month": wettest,
            "dry_months": int(dry),
            "planting_window": window
        })

print(f"Successfully wrote {len(counties)} rows to '{csv_filename}'.\n")
print("File Contents:")
with open(csv_filename, mode="r", encoding="utf-8") as f:
    print(f.read())

Successfully wrote 5 rows to 'rainfall_summary.csv'.

File Contents:
county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,Apr,4,Apr-May
Nakuru,916,Apr,3,Apr-May
Kisumu,1371,Apr,0,Apr-May
Mombasa,1075,May,2,Apr-May
Machakos,811,Nov,6,Nov-Dec



# Planting Advisory

1. **Long-Rains Sowing Window for Rift, Western, Central, and Coast (Nairobi, Nakuru, Kisumu, Mombasa):** Farmers in Nairobi, Nakuru, Kisumu, and Mombasa should conduct land preparation in February and March to commence sowing promptly during the **April–May** window. This two-month window captures peak annual rainfall—ranging from 265 mm in Nakuru and 390 mm in Mombasa to 395 mm in Nairobi and 420 mm in Kisumu—ensuring adequate soil moisture for seed germination and early vegetative development.
2. **Distinct Short-Rains Window for Machakos County:** In contrast to the other four counties, Machakos receives its highest precipitation during the short rains, establishing **November–December** as its primary planting window with 278 mm of combined rainfall and an individual peak of 182 mm in November. Because Machakos endures 6 dry months (< 50 mm), farmers should prioritize drought-tolerant, fast-maturing crops (such as pigeon peas, sorghum, cowpeas, and dryland maize) alongside soil moisture retention techniques.
3. **Cropping Intensity Guidance:** Kisumu enjoys abundant annual precipitation (1,371 mm) with zero dry months (< 50 mm), offering exceptional opportunities for double-cropping and water-intensive staple production. Conversely, Nakuru receives 916 mm across 3 dry months and Mombasa receives 1,075 mm across 2 dry months, indicating that post-planting mulching and water-conservation techniques are advisable to buffer against dry spells.

# Data Quality Notes

- **Manual Entry Errors & Imputation in Machakos Record:** The raw Machakos dataset suffered from manual transcription errors, specifically a non-numeric string placeholder (`"N/A"`) in April and a blank entry (`""`) in September. These missing values were imputed using the rounded arithmetic average of valid Machakos entries (**68 mm**), following standard observational data hygiene.
- **Impact of April Imputation on Agro-Climatic Seasonality:** Because April is historically the peak of the East African Long Rains, imputing 68 mm (the overall valid-month mean) likely underestimates true April precipitation in Machakos. Extension officers should note that if actual April rainfall was higher, Machakos may possess a viable secondary planting window in April–May that is obscured by the conservative imputation.
- **Point-Station Representation Caveat:** Weather station readings represent single geographical coordinates and may not fully reflect localized elevation gradients and microclimates (such as the Mau Forest influence in Nakuru or coastal rain bands in Mombasa).
- **Operational Recommendation:** This planting advisory serves as a strategic seasonal planning guide. The NGO should advise smallholder farmers to verify these baseline windows against real-time 7-day weather forecasts and advisories issued by the Kenya Meteorological Department (KMD) before sowing.